# Step 1: Housekeeping
Set up the required libraries and define the file paths for the CSV files and the SQLite database.

In [1]:
import sqlite3
import csv
import urllib.request
import os

# GitHub raw URLs for the three CSV files
# Replace the placeholder links below with the actual raw GitHub links from your course repo
BOOK_URL = "https://raw.githubusercontent.com/ab-fatima05/CIS-3120/refs/heads/main/Book.csv"
MEMBER_URL = "https://raw.githubusercontent.com/ab-fatima05/CIS-3120/refs/heads/main/Member.csv"
LOAN_URL = "https://raw.githubusercontent.com/ab-fatima05/CIS-3120/refs/heads/main/Loan.csv"

# Local file paths in Colab
BOOK_PATH = "/content/Book.csv"
MEMBER_PATH = "/content/Member.csv"
LOAN_PATH = "/content/Loan.csv"

# Database file path
DB_PATH = "/content/library.db"

print("Libraries imported and paths set.")

Libraries imported and paths set.


# Step 2: Download CSV Files
Download the Book, Member, and Loan CSV files from GitHub so they can be used in this notebook.

In [2]:
for url, path in [(BOOK_URL, BOOK_PATH), (MEMBER_URL, MEMBER_PATH), (LOAN_URL, LOAN_PATH)]:
    urllib.request.urlretrieve(url, path)
    print(f"Downloaded: {path}")

Downloaded: /content/Book.csv
Downloaded: /content/Member.csv
Downloaded: /content/Loan.csv


# Reset Database Before Reloading Data
Reset the database before rebuilding it. Without this, rerunning the notebook caused duplicate entry errors during the data loading step.

In [3]:
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
    print("Existing database removed.")
else:
    print("No existing database found. Starting fresh.")

Existing database removed.


# Step 3: Create Database and Tables
Create a connection to the SQLite database, enable foreign keys, and define the Book, Member, and Loan tables.

In [4]:
# Connect to database
connection = sqlite3.connect(DB_PATH)
cursor = connection.cursor()

# Enable foreign key support
cursor.execute("PRAGMA foreign_keys = ON;")

# Create tables
create_book_table = """
CREATE TABLE IF NOT EXISTS Book (
    callNo TEXT NOT NULL,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    PRIMARY KEY (callNo)
);
"""

create_member_table = """
CREATE TABLE IF NOT EXISTS Member (
    id INTEGER NOT NULL,
    firstname TEXT NOT NULL,
    lastName TEXT NOT NULL,
    PRIMARY KEY (id)
);
"""

create_loan_table = """
CREATE TABLE IF NOT EXISTS Loan (
    callNo TEXT NOT NULL,
    id INTEGER NOT NULL,
    dateBorrowed TEXT NOT NULL,
    dateReturned TEXT,
    dateDue TEXT NOT NULL,
    PRIMARY KEY (callNo, id, dateBorrowed),
    FOREIGN KEY (callNo) REFERENCES Book(callNo),
    FOREIGN KEY (id) REFERENCES Member(id)
);
"""

# Execute table creation
cursor.execute(create_book_table)
cursor.execute(create_member_table)
cursor.execute(create_loan_table)

# Save Changes
connection.commit()

# Confirm tables exist
tables = cursor.execute(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;"
).fetchall()

print("Tables successfully created.")
print("Tables in database:", tables)

Tables successfully created.
Tables in database: [('Book',), ('Loan',), ('Member',)]


# Step 4: Load Book Data
Go through the Book.csv file and insert each book into the Book table.

In [5]:
with open(BOOK_PATH, newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)

    for book_row in reader:
        cursor.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (book_row['callNo'], book_row['title'], book_row['author'])
        )

connection.commit()

book_total = cursor.execute("SELECT COUNT(*) FROM Book;").fetchone()[0]
print("Book rows loaded:", book_total)

Book rows loaded: 11


# Step 5: Load Member Data
Go through the Member.csv file and insert each member into the Member table.

In [6]:
with open(MEMBER_PATH, newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)

    for member_row in reader:
        cursor.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(member_row['id']), member_row['firstname'], member_row['lastName'])
        )

connection.commit()

member_total = cursor.execute("SELECT COUNT(*) FROM Member;").fetchone()[0]
print("Member rows loaded:", member_total)

Member rows loaded: 4


# Step 6: Load Loan Data
Go through the Loan.csv file, fix any missing return dates, and add each loan record into the Loan table.

In [7]:
with open(LOAN_PATH, newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)

    for loan_row in reader:
        returned_date = loan_row['dateReturned'] if loan_row['dateReturned'].strip() else None

        cursor.execute(
            "INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue) VALUES (?, ?, ?, ?, ?);",
            (
                loan_row['callNo'],
                int(loan_row['id']),
                loan_row['dateBorrowed'],
                returned_date,
                loan_row['dateDue']
            )
        )

connection.commit()

loan_total = cursor.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0]
print("Loan rows loaded:", loan_total)

Loan rows loaded: 4


# **SQL QUERIES:**



# Query 1 — All Books
Retrieve all columns from the Book table, ordered alphabetically by author's last name.

In [8]:
query_all_books = """
SELECT *
FROM Book
ORDER BY author;
"""

result = cursor.execute(query_all_books)

print("All Books (ordered by author):")
for book in result:
    print(book)

All Books (ordered by author):
('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


# Query 2 — Books Not Yet Returned
Retrieve the title of each book, and the first and last name of the member who borrowed it, for all loans where dateReturned is NULL.

In [9]:
query_unreturned_books = """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo = Book.callNo
JOIN Member ON Loan.id = Member.id
WHERE Loan.dateReturned IS NULL;
"""

result = cursor.execute(query_unreturned_books)

print("Books not yet returned:")
for record in result:
    print(record)

Books not yet returned:
("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


# Query 3 — Loan History for a Specific Book
Retrieve the full loan history for the book with callNo R 141 E45 2006, including the member’s name, when it was borrowed, when it was due, and when it was returned. Sort the results by the borrow date.

In [10]:
query_book_history = """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id = Member.id
WHERE Loan.callNo = 'R 141 E45 2006'
ORDER BY Loan.dateBorrowed ASC;
"""

result = cursor.execute(query_book_history)

print("Loan history for selected book:")
for entry in result:
    print(entry)

Loan history for selected book:
('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


# Query 4 — Members Who Have Never Borrowed a Book
Retrieve the id, firstname, and lastName of every member who does not appear in the Loan table.

In [11]:
query_no_loans = """
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
WHERE Loan.id IS NULL;
"""

result = cursor.execute(query_no_loans)

print("Members with no borrowing history:")
for member in result:
    print(member)

Members with no borrowing history:
(4, 'John', 'Martin')


# Query 5 — Count of Loans per Member
Retrieve each member's full name and the total number of loans they have made (including completed ones). Include members with zero loans. Order by number of loans descending.

In [12]:
query_loan_counts = """
SELECT Member.firstname, Member.lastName, COUNT(Loan.id) AS total_loans
FROM Member
LEFT JOIN Loan ON Member.id = Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
"""

result = cursor.execute(query_loan_counts)

print("Loan count for each member:")
for member_record in result:
    print(member_record)

Loan count for each member:
('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


# Query 6 — Book Usage
This query looks at how frequently each book appears in the loan records. It helps highlight which books are used more often compared to others in this dataset.

In [13]:
book_usage_query = """
SELECT
    Book.title,
    COUNT(Loan.callNo) AS borrow_count
FROM Book
LEFT JOIN Loan
    ON Book.callNo = Loan.callNo
GROUP BY Book.callNo, Book.title
ORDER BY borrow_count DESC, Book.title;
"""

results = cursor.execute(book_usage_query)

print("Book usage summary:")
for item in results:
    print(item)

Book usage summary:
('Medieval medicine and the plague', 2)
("Joe Celko's SQL puzzles & answers", 1)
('Medieval women', 1)
('Atlas of medieval Europe', 0)
('Data model patterns : conventions of thought', 0)
('Information modeling and relational databases', 0)
("Joe Celko's Trees and hierarchies in SQL for smarties", 0)
("Joe Celko's data & databases : concepts in practice", 0)
('Medicine in medieval England.', 0)
('Medieval miscellany', 0)
('Programming Android', 0)


# Brief Summary

One thing I noticed while working with this dataset is that the `dateReturned` field is sometimes left blank, so it had to be handled carefully and stored as NULL in the database. Another thing I ran into was that when rerunning the notebook, the database kept the previous data, which caused duplicate entry errors. Because of that, I had to reset the database before loading the data again to make sure everything runs properly from scratch. Also, the dataset itself is very small, which made it easier to work with but doesn’t really reflect how complex a real library system would be. For example, there’s no information about things like book categories, multiple copies of the same book, fines, or even basic member details beyond names. Because of that, the data is useful for learning the basics, but it would need a lot more detail to be practical in a real-world setting.

#Close the Database Connection

In [14]:
connection.close()
print("Connection closed. All operations complete.")

Connection closed. All operations complete.
